# qchem_colab_app v0.1.1 — 量子化学計算プロトタイプ (HF / 3-21G)

Google Colab上で構造最適化・振動数計算を行うプロトタイプです。
ColabReaction ( https://github.com/BILAB/ColabReaction ) と同様、
**上から順にセルを実行**していく構成になっています。

- **I. Setup Section** — 環境構築(このプロトタイプのコードとライブラリの準備)
- **II. Execution Section** — 実際の計算画面(GUI)

初めてJupyter/Colabを使う方へ: 各コードセルの左側にマウスを合わせると再生ボタン(▶)が
表示されます。それをクリックするか、セルを選択して `Shift+Enter` を押すと、そのセルが実行されます。
**必ず上のセルから順番に**実行してください(途中を飛ばすとエラーになります)。

## I. Setup Section

### 0. (推奨) Colabのランタイムバージョンを固定する

上部メニューの **「ランタイム」→「ランタイムのタイプを変更」** を開き、
**Runtime version** のプルダウンから特定のバージョン(例: `2026.07`)を選択してください。
これにより、Googleが将来Colabの基盤(Python本体やnumpy等)を更新しても、
このノートブックの動作が変わらないようにできます
(公式情報: https://research.google.com/colaboratory/runtime-version-faq.html )。

未設定のまま進めても動作はしますが、「今日動いたのに来月動かなくなった」を防ぐため、
公開版として配布する際は設定を強く推奨します。

In [ ]:
# 現在のPython/OS環境を確認しておきます(トラブルシューティング時の記録用)
!python --version
!lsb_release -d 2>/dev/null || cat /etc/os-release | head -1

### 1. コード本体の取得(GitHubから)

このプロトタイプのコードは https://github.com/orgchem610/qchem_colab_app で管理しています。
以後、コードの変更は**すべてこのGitHubリポジトリ経由**で反映します。
下のセルは、まだクローンしていなければ `git clone` を、既にクローン済みなら
最新版を取得する `git pull` を自動で行います。

In [ ]:
import os

GITHUB_USER = "orgchem610"
GITHUB_REPO = "qchem_colab_app"
PROJECT_DIR = f"/content/{GITHUB_REPO}"

if not os.path.exists(PROJECT_DIR):
    print(f"{GITHUB_REPO} をクローンします...")
    !git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {PROJECT_DIR}
else:
    print(f"{PROJECT_DIR} は既に存在するため、最新版を取得します(git pull)...")
    !cd {PROJECT_DIR} && git pull

# 特定のバージョン(タグ)を明示的に使いたい場合は、上のセル実行後に
# 以下のようなセルを追加して実行してください(例: v0.1.0固定)。
# !cd $PROJECT_DIR && git checkout v0.1.0

In [ ]:
import sys
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
print("sys.pathに追加しました:", PROJECT_DIR)

### 2. 依存ライブラリのインストール

「バージョンを固定して将来にわたり動作を変えない」という方針に沿い、
`requirements-lock.txt` が既にあればそちらを、なければ `requirements.txt` を使って
インストールします。**初回実行後、`requirements-lock.txt` が自動生成されます。**
それをGit管理下に置いてコミットしておくと、以後は全く同じ組み合わせを再現できます
(詳細は `README.md` / `CHANGELOG.md` 参照)。

パッケージを1つずつインストールし、途中で失敗したものがあれば
**どのパッケージが・どんなエラーで失敗したか**が分かるように表示します
(1本の `pip install -r ...` だけだと、どれが原因か分かりにくいためです)。

In [ ]:
import os
import subprocess

req_lock = os.path.join(PROJECT_DIR, "requirements-lock.txt")
req_plain = os.path.join(PROJECT_DIR, "requirements.txt")
req_file = req_lock if os.path.exists(req_lock) else req_plain
print(f"インストールに使用するファイル: {req_file}")

with open(req_file) as f:
    lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]

failed = []
for pkg in lines:
    print(f"--- installing: {pkg} ---")
    result = subprocess.run(
        ["pip", "install", "-q", pkg], capture_output=True, text=True)
    if result.returncode != 0:
        failed.append(pkg)
        print(f"❌ {pkg} のインストールに失敗しました。エラー末尾:")
        print("\n".join(result.stderr.strip().splitlines()[-15:]))
    else:
        print(f"OK: {pkg}")

if failed:
    print("\n=== 以下のパッケージのインストールに失敗しました ===")
    for pkg in failed:
        print(" -", pkg)
    print(
        "\n上のエラー内容(特に最後の数行)を確認してください。\n"
        "よくある原因は「そのパッケージがこのPythonバージョン用のビルド済み\n"
        "パッケージ(wheel)を提供しておらず、ソースからのビルドに失敗している」\n"
        "ことです。README.md のトラブルシューティング、または表示された\n"
        "パッケージ名とエラー内容をそのままご連絡ください。"
    )
else:
    print("\n全てのパッケージのインストールに成功しました。")

In [ ]:
# ---- GPU4PySCFの導入を試みる(ベストエフォート) ----
# 失敗しても後続処理には影響しません(qcapp/engine.pyが自動的にCPU実行へフォールバックします)。
import subprocess

try:
    # CPUのみのランタイムでは nvidia-smi コマンド自体が存在せず、
    # (存在すればreturncodeで判定できるはずが)FileNotFoundErrorが送出されるため、
    # 明示的に捕捉してGPUなし判定にフォールバックする。
    gpu_available = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
except FileNotFoundError:
    gpu_available = False
if gpu_available:
    print("GPUランタイムが検出されました。GPU4PySCFのインストールを試みます...")
    # CUDA12系ランタイムを想定した既定のパッケージ名です。
    # 導入に失敗する場合は、Colabの「ランタイム」→「ランタイムのタイプを変更」で
    # 表示されているCUDAバージョンを確認し、gpu4pyscf-cuda11x 等に読み替えてください。
    ret = subprocess.run(["pip", "install", "-q", "gpu4pyscf-cuda12x"], capture_output=True)
    if ret.returncode == 0:
        print("GPU4PySCFのインストールに成功しました。")
    else:
        print("GPU4PySCFのインストールに失敗しました。CPUで計算を続行します。")
        print(ret.stderr.decode()[-800:])
else:
    print("GPUランタイムが検出されませんでした(ランタイムのタイプがCPUのままか、GPU割り当てがありません)。")
    print("CPUで計算を実行します。GPUを使いたい場合は「ランタイム」→「ランタイムのタイプを変更」から")
    print("ハードウェアアクセラレータをGPU(T4等)に変更し、このSetup Sectionを最初から再実行してください。")

In [ ]:
# 今回、実際にインストールされたバージョン一式を記録します。
# これを requirements-lock.txt としてコミットしておくと、
# 「今日動いた環境」をそのまま将来に固定できます。
!pip freeze > $PROJECT_DIR/requirements-lock.txt
print("requirements-lock.txt を書き出しました。ダウンロードしてGitにコミットすることを推奨します。")

import importlib
pyscf = importlib.import_module("pyscf")
print("インストールされた pyscf のバージョン:", pyscf.__version__)

### 3. 動作確認用サンプル構造の準備(任意)

初めて実行する方は、いきなりご自身の分子(70原子級など)を試す前に、
下のセルで作成される小さなサンプル(エタノール, 9原子, C・H・Oのみ)で
一度パイプライン全体(構造最適化→振動数計算)が正常に動くことを確認することを強く推奨します。

In [ ]:
sample_xyz = """9
Ethanol (CH3CH2OH) - rough starting geometry for connectivity testing
C   0.000000   0.000000   0.000000
C   1.510000   0.000000   0.000000
O   1.970000   1.340000   0.000000
H  -0.363000   0.000000   1.028000
H  -0.363000  -0.890000  -0.514000
H  -0.363000   0.890000  -0.514000
H   2.020000  -0.360000   0.890000
H   2.020000  -0.360000  -0.890000
H   1.445000   2.144000   0.000000
"""
with open("/content/sample_ethanol.xyz", "w") as f:
    f.write(sample_xyz)
print("/content/sample_ethanol.xyz を作成しました。")
print("下の実行画面のアップロード欄でこのファイルを選択して、まず動作確認することをお勧めします。")
print("(Colab左側のフォルダアイコンから /content 内のファイルとして参照できます)")

## II. Execution Section

以下のセルを実行すると操作画面が表示されます。

1. 「① 構造ファイルと計算条件」で `.xyz` ファイルをアップロードし、条件を設定
2. 「② 実行」の **「計算を実行する」** ボタンを押す
3. ログに構造最適化・振動数計算それぞれの所要時間と、合計時間が表示されます
4. 「③ 可視化」でエネルギー推移・軌跡・振動アニメーション・分子軌道・電荷密度を確認
5. 「④ ダウンロード」から最終構造(ColabReactionにそのまま渡せる.xyz)・
   トラジェクトリ・moldenファイルを取得

**条件を変えて再実行したい場合は、このセルをもう一度実行し直してください
(GUI部品が再生成され、状態がリセットされます)。**

In [ ]:
import gui
from IPython.display import display

app = gui.build_app()
display(app)